# 01. 30봉 OHLCV 자금유입 스캘핑 전처리

완료된 `t`봉까지의 30개 1분봉으로 자금 유입을 표현하고, `first_ask[t+1]`에 진입했을 때 미래 각 완성봉의 `min_bid` 기준으로 비용 차감 후 +3%를 달성할 수 있는지를 생성한다.

- `t+1`의 완성 OHLCV는 feature로 사용하지 않는다.
- 진입·청산 quote는 label과 실행 수익 계산에만 사용한다.
- 절대 거래량은 보조 ablation feature이며 기본 모델 입력은 상대 거래량과 상대 거래대금이다.
- 당일 VWAP·고점·저점과 EMA는 각 행 시점까지의 데이터만 사용한다.
- Section 31: [SEC FY2026 advisory](https://www.sec.gov/rules-regulations/fee-rate-advisories/2026-2)
- FINRA TAF: [FINRA Schedule A Section 1](https://www.finra.org/rules-guidance/rulebooks/corporate-organization/section-1-member-regulatory-fees)

In [1]:
from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
DATA_ROOT = (PROJECT_ROOT / '../../data/stock_data/raw').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

VERSION = 'scalp_30m_ohlcv_net3_minbid_5m_v1'
SEQUENCE_LENGTH = 30
HORIZON_MINUTES = 5
NET_TARGET_RETURN = 0.03
STAKE_USD = 100.0
COMMISSION_USD_PER_SIDE = 0.0
SEC_SECTION31_RATE = 20.60 / 1_000_000
FINRA_TAF_PER_SHARE = 0.000166
FINRA_TAF_CAP_USD = 8.30

SOURCE_FILES = sorted(DATA_ROOT.rglob('*_enriched.csv'))
assert SOURCE_FILES, DATA_ROOT
print('project:', PROJECT_ROOT)
print('source files:', len(SOURCE_FILES))
print('result root:', RESULT_ROOT)

project: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
source files: 208
result root: /home/user/urbandatalab/YSLee/results/preprocessing


## Feature 정의

현재 봉의 거래량 급증은 현재 봉을 제외한 이전 5/10/20봉 median과 비교한다. 공격적 매수·매도 거래량은 총 거래량에 대한 비율로 바꿔 종목 크기의 영향을 줄인다.

In [2]:
REQUIRED_COLUMNS = [
    'symbol', 'timestamp_utc', 'open', 'high', 'low', 'close',
    'volume', 'trade_count', 'alpaca_vwap',
    'first_ask', 'last_bid', 'min_bid',
    'aggressive_buy_volume', 'aggressive_sell_volume', 'unknown_trade_volume',
]

PRICE_FEATURES = [
    'log_open_to_previous_close', 'log_high_to_open', 'log_low_to_open',
    'log_close_to_open', 'log_return_1', 'log_range',
    'upper_wick_ratio', 'lower_wick_ratio', 'close_location',
]
ABSOLUTE_VOLUME_FEATURES = ['log_volume']
RELATIVE_FLOW_FEATURES = [
    'log_relative_volume_5', 'log_relative_volume_10', 'log_relative_volume_20',
    'robust_log_volume_z20', 'volume_ema3_to_ema10',
    'log_relative_notional_20', 'log_relative_trade_count_20',
    'log_relative_average_trade_size_20',
    'aggressive_net_flow', 'aggressive_participation', 'aggressive_net_flow_3',
    'signed_log_relative_aggressive_net_notional_20',
    'log_relative_aggressive_gross_notional_20',
    'return_x_log_relative_volume_20',
]
LOCATION_FEATURES = [
    'close_to_bar_vwap', 'close_to_session_vwap',
    'close_to_ema5', 'close_to_ema9', 'close_to_ema20',
    'close_to_session_high', 'close_to_session_low',
    'close_to_prior_session_high', 'close_to_prior_session_low',
    'close_to_high_5', 'close_to_low_5',
    'close_to_high_10', 'close_to_low_10',
    'close_to_high_20', 'close_to_low_20',
]
FEATURE_NAMES = PRICE_FEATURES + ABSOLUTE_VOLUME_FEATURES + RELATIVE_FLOW_FEATURES + LOCATION_FEATURES
DEFAULT_FEATURE_NAMES = [name for name in FEATURE_NAMES if name not in ABSOLUTE_VOLUME_FEATURES]


def safe_log_ratio(numerator, denominator, epsilon=1e-8):
    return np.log(np.clip(numerator, epsilon, None) / np.clip(denominator, epsilon, None))


def prior_median(values, window):
    return values.shift(1).rolling(window, min_periods=window).median()


def relative_log(values, window):
    baseline = prior_median(values, window)
    return np.log((values.clip(lower=0) + 1.0) / (baseline.clip(lower=0) + 1.0))


def build_features(frame):
    frame = frame.copy()
    open_price = frame['open'].astype(float)
    high = frame['high'].astype(float)
    low = frame['low'].astype(float)
    close = frame['close'].astype(float)
    volume = frame['volume'].astype(float).clip(lower=0)
    trade_count = frame['trade_count'].astype(float).clip(lower=0)
    bar_vwap = frame['alpaca_vwap'].astype(float)
    previous_close = close.shift(1)

    features = pd.DataFrame(index=frame.index)
    features['log_open_to_previous_close'] = safe_log_ratio(open_price, previous_close)
    features['log_high_to_open'] = safe_log_ratio(high, open_price)
    features['log_low_to_open'] = safe_log_ratio(low, open_price)
    features['log_close_to_open'] = safe_log_ratio(close, open_price)
    features['log_return_1'] = safe_log_ratio(close, previous_close)
    features['log_range'] = safe_log_ratio(high, low)
    features['upper_wick_ratio'] = (high - np.maximum(open_price, close)) / open_price.clip(lower=1e-8)
    features['lower_wick_ratio'] = (np.minimum(open_price, close) - low) / open_price.clip(lower=1e-8)
    features['close_location'] = (close - low) / (high - low).replace(0, np.nan)
    features['close_location'] = features['close_location'].fillna(0.5)

    log_volume = np.log1p(volume)
    features['log_volume'] = log_volume
    for window in [5, 10, 20]:
        features[f'log_relative_volume_{window}'] = relative_log(volume, window)
    prior_log_median = log_volume.shift(1).rolling(20, min_periods=20).median()
    prior_log_q25 = log_volume.shift(1).rolling(20, min_periods=20).quantile(0.25)
    prior_log_q75 = log_volume.shift(1).rolling(20, min_periods=20).quantile(0.75)
    robust_scale = (prior_log_q75 - prior_log_q25).clip(lower=0.05)
    features['robust_log_volume_z20'] = (log_volume - prior_log_median) / robust_scale
    volume_ema3 = volume.ewm(span=3, adjust=False).mean()
    volume_ema10 = volume.ewm(span=10, adjust=False).mean()
    features['volume_ema3_to_ema10'] = safe_log_ratio(volume_ema3 + 1, volume_ema10 + 1)

    notional = bar_vwap * volume
    average_trade_size = volume / trade_count.clip(lower=1)
    features['log_relative_notional_20'] = relative_log(notional, 20)
    features['log_relative_trade_count_20'] = relative_log(trade_count, 20)
    features['log_relative_average_trade_size_20'] = relative_log(average_trade_size, 20)

    aggressive_buy = frame['aggressive_buy_volume'].astype(float).clip(lower=0)
    aggressive_sell = frame['aggressive_sell_volume'].astype(float).clip(lower=0)
    classified_volume = aggressive_buy + aggressive_sell
    features['aggressive_net_flow'] = ((aggressive_buy - aggressive_sell) / classified_volume.replace(0, np.nan)).fillna(0)
    features['aggressive_participation'] = (classified_volume / volume.replace(0, np.nan)).fillna(0).clip(0, 1)
    flow3 = (aggressive_buy - aggressive_sell).rolling(3, min_periods=1).sum()
    classified3 = classified_volume.rolling(3, min_periods=1).sum()
    features['aggressive_net_flow_3'] = (flow3 / classified3.replace(0, np.nan)).fillna(0)
    prior_notional_median20 = prior_median(notional, 20).clip(lower=1e-8)
    aggressive_net_notional = bar_vwap * (aggressive_buy - aggressive_sell)
    aggressive_gross_notional = bar_vwap * classified_volume
    relative_net_notional = aggressive_net_notional / prior_notional_median20
    features['signed_log_relative_aggressive_net_notional_20'] = (
        np.sign(relative_net_notional) * np.log1p(relative_net_notional.abs())
    )
    features['log_relative_aggressive_gross_notional_20'] = np.log1p(
        aggressive_gross_notional / prior_notional_median20
    )
    features['return_x_log_relative_volume_20'] = (
        features['log_return_1'] * features['log_relative_volume_20']
    )

    cumulative_volume = volume.cumsum()
    session_vwap = (bar_vwap * volume).cumsum() / cumulative_volume.replace(0, np.nan)
    features['close_to_bar_vwap'] = close / bar_vwap.clip(lower=1e-8) - 1
    features['close_to_session_vwap'] = close / session_vwap.clip(lower=1e-8) - 1
    for span in [5, 9, 20]:
        ema = close.ewm(span=span, adjust=False).mean()
        features[f'close_to_ema{span}'] = close / ema.clip(lower=1e-8) - 1

    session_high = high.cummax()
    session_low = low.cummin()
    prior_session_high = high.shift(1).cummax()
    prior_session_low = low.shift(1).cummin()
    features['close_to_session_high'] = close / session_high.clip(lower=1e-8) - 1
    features['close_to_session_low'] = close / session_low.clip(lower=1e-8) - 1
    features['close_to_prior_session_high'] = close / prior_session_high.clip(lower=1e-8) - 1
    features['close_to_prior_session_low'] = close / prior_session_low.clip(lower=1e-8) - 1
    for window in [5, 10, 20]:
        rolling_high = high.rolling(window, min_periods=window).max()
        rolling_low = low.rolling(window, min_periods=window).min()
        features[f'close_to_high_{window}'] = close / rolling_high.clip(lower=1e-8) - 1
        features[f'close_to_low_{window}'] = close / rolling_low.clip(lower=1e-8) - 1

    return features[FEATURE_NAMES].astype(np.float32)


print('feature count:', len(FEATURE_NAMES))
print('default feature count:', len(DEFAULT_FEATURE_NAMES))

feature count: 39
default feature count: 38


## 실행 가능 net +3% label

매수 ask와 각 미래 완성봉의 최저 bid인 `min_bid`를 사용한다. 따라서 봉 안에서 잠깐 +3%를 터치한 경우가 아니라, 해당 분의 가장 불리한 bid로도 net +3%가 유지된 경우만 positive다. 매도 시 Section 31과 FINRA TAF를 차감한다. 브로커별 별도 수수료가 있으면 아래 설정을 변경해야 한다.

In [3]:
def executable_net_return(entry_ask, exit_bid):
    shares = STAKE_USD / entry_ask
    exit_notional = shares * exit_bid
    section31 = exit_notional * SEC_SECTION31_RATE
    taf = 0.0 if exit_bid < FINRA_TAF_PER_SHARE else min(shares * FINRA_TAF_PER_SHARE, FINRA_TAF_CAP_USD)
    total_cost = 2 * COMMISSION_USD_PER_SIDE + section31 + taf
    return (exit_notional - STAKE_USD - total_cost) / STAKE_USD


def load_source(path):
    frame = pd.read_csv(path, usecols=REQUIRED_COLUMNS)
    frame['timestamp_utc'] = pd.to_datetime(frame['timestamp_utc'], utc=True, errors='coerce')
    numeric_columns = [column for column in REQUIRED_COLUMNS if column not in {'symbol', 'timestamp_utc'}]
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors='coerce')
    return frame.sort_values('timestamp_utc').reset_index(drop=True)


sequences = []
metadata_records = []
quality_records = []
started = time.time()
sample_index = 0

for file_number, path in enumerate(SOURCE_FILES, start=1):
    frame = load_source(path)
    features = build_features(frame)
    timestamps = frame['timestamp_utc']
    minute_number = timestamps.astype('int64').to_numpy() // 60_000_000_000
    run_number = np.r_[0, np.cumsum(np.diff(minute_number) != 1)]
    feature_matrix = features.to_numpy(np.float32)
    total_candidates = 0
    valid_candidates = 0
    rejected_gap = 0
    rejected_feature = 0
    rejected_quote = 0

    for decision_index in range(SEQUENCE_LENGTH - 1, len(frame) - HORIZON_MINUTES):
        total_candidates += 1
        full_start = decision_index - SEQUENCE_LENGTH + 1
        full_end = decision_index + HORIZON_MINUTES + 1
        if not np.all(np.diff(minute_number[full_start:full_end]) == 1):
            rejected_gap += 1
            continue
        sequence = feature_matrix[full_start:decision_index + 1]
        if sequence.shape != (SEQUENCE_LENGTH, len(FEATURE_NAMES)) or not np.isfinite(sequence).all():
            rejected_feature += 1
            continue

        entry_index = decision_index + 1
        future_indices = np.arange(entry_index, entry_index + HORIZON_MINUTES)
        entry_ask = float(frame.at[entry_index, 'first_ask'])
        future_min_bid = frame.loc[future_indices, 'min_bid'].to_numpy(float)
        timeout_bid = float(frame.at[future_indices[-1], 'last_bid'])
        if (not np.isfinite(entry_ask) or entry_ask <= 0 or
                not np.isfinite(future_min_bid).all() or np.any(future_min_bid <= 0) or
                not np.isfinite(timeout_bid) or timeout_bid <= 0):
            rejected_quote += 1
            continue

        horizon_returns = np.asarray([
            executable_net_return(entry_ask, exit_bid) for exit_bid in future_min_bid
        ], dtype=np.float32)
        reached = horizon_returns >= NET_TARGET_RETURN
        first_hit_minute = int(np.flatnonzero(reached)[0] + 1) if reached.any() else 0
        timeout_net_return = float(executable_net_return(entry_ask, timeout_bid))
        decision_bar_timestamp = timestamps.iloc[decision_index]
        decision_available_timestamp = decision_bar_timestamp + pd.Timedelta(minutes=1)

        sequences.append(sequence.copy())
        record = {
            'sample_id': f'{VERSION}_{sample_index:08d}',
            'feature_index': sample_index,
            'session': path.parent.name,
            'symbol': str(frame.at[decision_index, 'symbol']),
            'source_file': str(path.resolve()),
            'run_id': f'{path.stem}::{int(run_number[decision_index])}',
            'input_start_timestamp': timestamps.iloc[full_start],
            'decision_bar_timestamp': decision_bar_timestamp,
            'decision_available_timestamp': decision_available_timestamp,
            'entry_timestamp': timestamps.iloc[entry_index],
            'entry_ask': entry_ask,
            'target_net3_within_5m': int(reached.any()),
            'first_hit_minute': first_hit_minute,
            'best_min_bid_net_return_1_5m': float(horizon_returns.max()),
            'timeout_bid_5m': timeout_bid,
            'timeout_net_return_5m': timeout_net_return,
        }
        for horizon, (future_index, exit_bid, net_return) in enumerate(
            zip(future_indices, future_min_bid, horizon_returns), start=1
        ):
            record[f'future_timestamp_{horizon}'] = timestamps.iloc[future_index]
            record[f'min_bid_{horizon}'] = float(exit_bid)
            record[f'net_return_min_bid_{horizon}'] = float(net_return)
        metadata_records.append(record)
        sample_index += 1
        valid_candidates += 1

    quality_records.append({
        'session': path.parent.name, 'source_file': str(path.resolve()),
        'symbol': str(frame['symbol'].iloc[0]) if len(frame) else '',
        'rows': len(frame), 'total_candidates': total_candidates,
        'valid_candidates': valid_candidates, 'rejected_gap': rejected_gap,
        'rejected_feature': rejected_feature, 'rejected_quote': rejected_quote,
    })
    if file_number % 25 == 0 or file_number == len(SOURCE_FILES):
        print(f'{file_number:>3}/{len(SOURCE_FILES)} files | samples={sample_index:,}')

print(f'elapsed: {(time.time() - started) / 60:.2f} min')

 25/208 files | samples=9,116


 50/208 files | samples=17,281


 75/208 files | samples=26,050


100/208 files | samples=33,019


125/208 files | samples=42,705


150/208 files | samples=51,108


175/208 files | samples=61,498


200/208 files | samples=69,960


208/208 files | samples=72,181
elapsed: 0.53 min


## Artifact 저장과 기본 감사

In [4]:
sequence_array = np.stack(sequences).astype(np.float32)
metadata = pd.DataFrame(metadata_records)
quality = pd.DataFrame(quality_records)
assert len(sequence_array) == len(metadata)
assert sequence_array.shape[1:] == (SEQUENCE_LENGTH, len(FEATURE_NAMES))
assert np.isfinite(sequence_array).all()
assert metadata['feature_index'].equals(pd.Series(np.arange(len(metadata)), name='feature_index'))
assert (metadata['decision_available_timestamp'] == metadata['entry_timestamp']).all()

feature_path = RESULT_ROOT / f'{VERSION}_features.npz'
metadata_path = RESULT_ROOT / f'{VERSION}_metadata.parquet'
quality_path = RESULT_ROOT / f'{VERSION}_quality.parquet'
distribution_path = RESULT_ROOT / f'{VERSION}_label_distribution.parquet'
schema_path = RESULT_ROOT / f'{VERSION}_schema.json'

np.savez_compressed(feature_path, sequence=sequence_array)
metadata.to_parquet(metadata_path, index=False, compression='zstd')
quality.to_parquet(quality_path, index=False, compression='zstd')
distribution = metadata.groupby('session').agg(
    samples=('sample_id', 'size'),
    symbols=('symbol', 'nunique'),
    positives=('target_net3_within_5m', 'sum'),
    positive_rate=('target_net3_within_5m', 'mean'),
    mean_best_min_bid_net_return=('best_min_bid_net_return_1_5m', 'mean'),
    mean_timeout_net_return=('timeout_net_return_5m', 'mean'),
).reset_index()
distribution.to_parquet(distribution_path, index=False, compression='zstd')

schema = {
    'version': VERSION,
    'source_glob': str(DATA_ROOT / 'session_*/*_enriched.csv'),
    'sequence_length': SEQUENCE_LENGTH,
    'feature_names': FEATURE_NAMES,
    'default_feature_names': DEFAULT_FEATURE_NAMES,
    'ablation_only_features': ABSOLUTE_VOLUME_FEATURES,
    'feature_groups': {
        'price': PRICE_FEATURES, 'absolute_volume': ABSOLUTE_VOLUME_FEATURES,
        'relative_flow': RELATIVE_FLOW_FEATURES, 'location': LOCATION_FEATURES,
    },
    'timing': {
        'decision': 'immediately after completed t bar',
        'entry': 'first_ask[t+1]',
        'feature_cutoff': 'all model features end at completed t bar',
    },
    'label': {
        'name': 'target_net3_within_5m',
        'positive_rule': 'any completed-minute min_bid from t+1 through t+5 yields net return >= 3%',
        'timeout': 'last_bid[t+5]',
    },
    'execution_costs': {
        'stake_usd': STAKE_USD, 'commission_usd_per_side': COMMISSION_USD_PER_SIDE,
        'section31_rate_per_sold_dollar': SEC_SECTION31_RATE,
        'finra_taf_per_sold_share': FINRA_TAF_PER_SHARE,
        'finra_taf_cap_usd': FINRA_TAF_CAP_USD,
        'spread': 'captured by entry ask and exit bid',
    },
    'scaling': 'not applied; fit on each future Train fold only',
    'artifacts': {
        'features': str(feature_path), 'metadata': str(metadata_path),
        'quality': str(quality_path), 'label_distribution': str(distribution_path),
    },
}
schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')

print('sequence:', sequence_array.shape)
print('metadata:', metadata.shape)
print('positive rate:', f"{metadata['target_net3_within_5m'].mean():.2%}")
print('default features:', len(DEFAULT_FEATURE_NAMES), '/', len(FEATURE_NAMES))
display(distribution)
display(quality[['rows', 'total_candidates', 'valid_candidates', 'rejected_gap', 'rejected_feature', 'rejected_quote']].sum().to_frame('count'))

sequence: (72181, 30, 39)
metadata: (72181, 31)
positive rate: 3.94%
default features: 38 / 39


,session,samples,symbols,positives,positive_rate,mean_best_min_bid_net_return,mean_timeout_net_return
0,session_2026-07-07,4321,13,115,0.026614,-0.002089,-0.004694
1,session_2026-07-08,10343,25,362,0.035000,-0.003138,-0.007469
2,session_2026-07-09,10208,27,388,0.038009,-0.001989,-0.006434
3,session_2026-07-10,10230,29,534,0.052199,-0.003133,-0.008216
4,session_2026-07-13,7953,18,405,0.050924,-0.001929,-0.006242
5,session_2026-07-14,7043,20,266,0.037768,-0.003032,-0.008038
6,session_2026-07-15,9556,21,409,0.042800,-0.002456,-0.007069
7,session_2026-07-16,5918,16,237,0.040047,-0.001598,-0.005094
8,session_2026-07-17,6609,20,131,0.019821,-0.003383,-0.006528


,count
rows,127423
total_candidates,120351
valid_candidates,72181
rejected_gap,45420
rejected_feature,1276
rejected_quote,1474


In [5]:
label_summary = pd.Series({
    'samples': len(metadata),
    'symbols': metadata['symbol'].nunique(),
    'sessions': metadata['session'].nunique(),
    'positive_samples': int(metadata['target_net3_within_5m'].sum()),
    'positive_rate': metadata['target_net3_within_5m'].mean(),
    'median_first_hit_positive': metadata.loc[metadata['target_net3_within_5m'].eq(1), 'first_hit_minute'].median(),
    'mean_best_min_bid_net_return': metadata['best_min_bid_net_return_1_5m'].mean(),
    'mean_timeout_net_return': metadata['timeout_net_return_5m'].mean(),
})
display(label_summary.to_frame('value'))
display(pd.crosstab(metadata['session'], metadata['first_hit_minute']))

,value
samples,72181.000000
symbols,111.000000
sessions,9.000000
positive_samples,2847.000000
positive_rate,0.039443
median_first_hit_positive,4.000000
mean_best_min_bid_net_return,-0.002574
mean_timeout_net_return,-0.006849


first_hit_minute,0,2,3,4,5
session,,,,,
session_2026-07-07,4206,12,27,40,36
session_2026-07-08,9981,38,87,122,115
session_2026-07-09,9820,64,109,102,113
session_2026-07-10,9696,90,149,145,150
session_2026-07-13,7548,69,123,112,101
session_2026-07-14,6777,46,73,80,67
session_2026-07-15,9147,57,99,131,122
session_2026-07-16,5681,27,63,71,76
session_2026-07-17,6478,10,36,41,44
